In [ ]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 3 – Location problems
#  Section: Exercise 2
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP            # Modeling language
using HiGHS           # Solver
using CSV             # For reading CSV files
using DataFrames      # For handling DataFrame operations
using Distances       # Distance computations

# Importing Python libraries for plotting
include("utils/pmp-fault_utils.jl")

# Function to solve the fault-tolerant p-median problem
function solve_fault_tolerant_p_median(file_path; p = 3, r = 2)

    # Load latitude and longitude data
    coordinates = CSV.read(file_path, header=true, DataFrame) |> Matrix{Float64}

    # Compute Haversine distance matrix
    distance_matrix = Distances.pairwise(Haversine(), coordinates, dims=1)

    # Number of possible facility / client locations
    n = size(distance_matrix, 1)

    # Create the model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # x[i, j] = 1 if node j is assigned to facility at node i
    @variable(model, x[1:n, 1:n], Bin)

    # Objective: minimize total distance
    @objective(model, Min, sum(distance_matrix[i, j] * x[i, j] for i in 1:n, j in 1:n))

    # Exactly p medians: sum of diagonal entries
    @constraint(model, sum(x[i,i] for i in 1:n) == p)

    # Each node j must be covered r times (including possibly by itself)
    @constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n) + x[j, j] == r)

    # Assignments only allowed to open facilities
    @constraint(model, [i=1:n, j=1:n], x[i, j] <= x[i, i])

    # Solve the model
    JuMP.optimize!(model)

    # Extract solution
    assignments = Dict()
    for i in 1:n
        if JuMP.value(x[i,i]) > 0.5
            assignments[i] = [j for j in 1:n if JuMP.value(x[i,j]) > 0.5]
        end
    end

    # Display solution
    println("Objective : ", JuMP.objective_value(model))
    for (facility, clients) in assignments
        println("Facility: $facility | Clients: $clients")
    end

    # Plot the solution
    map = plot_solution(coordinates, assignments, p)
    display(map)
end

# Example usage
solve_fault_tolerant_p_median("data/sjc_coordinates.csv", p = 3, r = 2)

PyObject <folium.folium.Map object at 0x000002A2F68F03B0>

Objective : 61491.29434666567
Facility: 39 | Clients: [1, 3, 5, 6, 7, 8, 10, 13, 14, 17, 20, 22, 25, 27, 30, 34, 35, 36, 37, 39, 42, 45, 46, 50, 52, 54, 56, 58, 59, 62, 64, 66, 67, 68, 69, 70, 74, 75, 79, 80, 82, 86, 90, 92, 93, 94, 95, 96, 98, 99, 102]
Facility: 55 | Clients: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102]
Facility: 12 | Clients: [2, 4, 9, 11, 12, 15, 16, 18, 19, 21, 23, 24, 26, 28, 29, 31, 32, 33, 38, 40, 41, 43, 44, 47, 48, 49, 51, 53, 57, 60, 61, 63, 65, 71, 72, 73, 76, 77, 78, 81, 83, 84, 85, 87, 88, 89, 91, 97, 100, 101]
